<a href="https://colab.research.google.com/github/nicolasojda/bigdata-evaluacion-citibike/blob/main/notebooks/analisis_exploratorio_y_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Análisis exploratorio y modelo predictivo: NYC Citi Bike en BigQuery\n",
    "\n",
    "Este notebook ejecuta, desde Python, las consultas del análisis exploratorio y la evaluación de los modelos\n",
    "del proyecto. Todo el procesamiento ocurre en BigQuery; el notebook solo recibe resultados agregados.\n",
    "\n",
    "Para ejecutarlo en Google Colab basta con autenticarse con la cuenta que tiene acceso al proyecto\n",
    "`bigdata-evaluacion-2-511416`. Las consultas son las mismas de `sql/02_analisis_exploratorio.sql` y\n",
    "`sql/05_evaluacion_prediccion.sql`."
   ],
   "id": "c00"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "# Autenticación (solo necesaria en Google Colab)\n",
    "try:\n",
    "    from google.colab import auth\n",
    "    auth.authenticate_user()\n",
    "except ImportError:\n",
    "    pass  # fuera de Colab se usan las credenciales de gcloud"
   ],
   "id": "c01"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "from google.cloud import bigquery\n",
    "\n",
    "PROYECTO = \"bigdata-evaluacion-2-511416\"\n",
    "TABLA = f\"`{PROYECTO}.citibike_optimizada.viajes_particionados`\"\n",
    "PUBLICA = \"`bigquery-public-data.new_york_citibike.citibike_trips`\"\n",
    "MODELO_BASE = f\"`{PROYECTO}.citibike_optimizada.modelo_prediccion_duracion`\"\n",
    "MODELO_V2 = f\"`{PROYECTO}.citibike_optimizada.modelo_prediccion_duracion_v2`\"\n",
    "\n",
    "cliente = bigquery.Client(project=PROYECTO, location=\"US\")\n",
    "\n",
    "def consulta(sql):\n",
    "    trabajo = cliente.query(sql)\n",
    "    df = trabajo.to_dataframe()\n",
    "    print(f\"Bytes procesados: {trabajo.total_bytes_processed / 1e9:.2f} GB\")\n",
    "    return df"
   ],
   "id": "c02"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Estadísticos generales de la duración\n",
    "\n",
    "La diferencia entre el promedio y la mediana muestra la asimetría de la distribución, y el conteo de viajes\n",
    "sobre 24 horas identifica los registros anómalos."
   ],
   "id": "c03"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "estadisticos = consulta(f\"\"\"\n",
    "SELECT\n",
    "  COUNT(*)                                        AS total_viajes,\n",
    "  ROUND(AVG(tripduration), 1)                     AS promedio_seg,\n",
    "  APPROX_QUANTILES(tripduration, 100)[OFFSET(50)] AS mediana_seg,\n",
    "  MAX(tripduration)                               AS maximo_seg,\n",
    "  COUNTIF(tripduration > 86400)                   AS viajes_sobre_24h\n",
    "FROM {TABLA}\n",
    "\"\"\")\n",
    "estadisticos"
   ],
   "id": "c04"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Viajes por hora de inicio y tipo de usuario\n",
    "\n",
    "Corresponde a la Figura 2 del informe técnico."
   ],
   "id": "c05"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "por_hora = consulta(f\"\"\"\n",
    "SELECT\n",
    "  EXTRACT(HOUR FROM starttime)     AS hora_inicio,\n",
    "  COUNTIF(usertype = 'Subscriber') AS viajes_abonados,\n",
    "  COUNTIF(usertype = 'Customer')   AS viajes_ocasionales\n",
    "FROM {PUBLICA}\n",
    "WHERE starttime BETWEEN '2016-01-01' AND '2018-05-31'\n",
    "  AND tripduration BETWEEN 60 AND 7200\n",
    "GROUP BY hora_inicio\n",
    "ORDER BY hora_inicio\n",
    "\"\"\")\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 4.5))\n",
    "ax.plot(por_hora.hora_inicio, por_hora.viajes_abonados / 1e6, marker=\"o\", label=\"Abonados (Subscriber)\")\n",
    "ax.plot(por_hora.hora_inicio, por_hora.viajes_ocasionales / 1e6, marker=\"o\", label=\"Ocasionales (Customer)\")\n",
    "ax.set_xlabel(\"Hora de inicio\")\n",
    "ax.set_ylabel(\"Viajes (millones)\")\n",
    "ax.set_xticks(range(24))\n",
    "ax.grid(alpha=0.3)\n",
    "ax.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ],
   "id": "c06"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Viajes por día de la semana\n",
    "\n",
    "En BigQuery, `DAYOFWEEK` toma el valor 1 para el domingo y 7 para el sábado. El filtro por `anio` usa la\n",
    "partición de la tabla optimizada."
   ],
   "id": "c07"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "por_dia = consulta(f\"\"\"\n",
    "SELECT\n",
    "  EXTRACT(DAYOFWEEK FROM starttime) AS dia_semana,\n",
    "  COUNT(*)                          AS viajes,\n",
    "  ROUND(AVG(tripduration), 1)       AS promedio_seg\n",
    "FROM {TABLA}\n",
    "WHERE anio >= 2016\n",
    "  AND tripduration BETWEEN 60 AND 7200\n",
    "GROUP BY dia_semana\n",
    "ORDER BY dia_semana\n",
    "\"\"\")\n",
    "nombres = [\"Dom\", \"Lun\", \"Mar\", \"Mié\", \"Jue\", \"Vie\", \"Sáb\"]\n",
    "fig, ax = plt.subplots(figsize=(8, 4))\n",
    "ax.bar([nombres[d - 1] for d in por_dia.dia_semana], por_dia.viajes / 1e6)\n",
    "ax.set_ylabel(\"Viajes (millones)\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "por_dia"
   ],
   "id": "c08"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Comparación de los modelos\n",
    "\n",
    "Sin un segundo argumento, `ML.EVALUATE` calcula las métricas sobre la porción de datos que BigQuery ML\n",
    "reservó para evaluación durante el entrenamiento."
   ],
   "id": "c09"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "comparacion = consulta(f\"\"\"\n",
    "SELECT 'Iteración 1 (base)' AS modelo, mean_absolute_error, mean_squared_error,\n",
    "       SQRT(mean_squared_error) AS rmse, median_absolute_error, r2_score\n",
    "FROM ML.EVALUATE(MODEL {MODELO_BASE})\n",
    "UNION ALL\n",
    "SELECT 'Iteración 2 (filtrado)', mean_absolute_error, mean_squared_error,\n",
    "       SQRT(mean_squared_error), median_absolute_error, r2_score\n",
    "FROM ML.EVALUATE(MODEL {MODELO_V2})\n",
    "\"\"\")\n",
    "comparacion.round(4)"
   ],
   "id": "c10"
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Inferencia sobre un viaje de ejemplo\n",
    "\n",
    "Viaje de un abonado entre las estaciones 519 y 492, iniciado un lunes a las 8:00."
   ],
   "id": "c11"
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "consulta(f\"\"\"\n",
    "SELECT predicted_tripduration,\n",
    "       tripduration AS duracion_real,\n",
    "       ABS(predicted_tripduration - tripduration) AS diferencia_absoluta\n",
    "FROM ML.PREDICT(MODEL {MODELO_V2},\n",
    "  (SELECT 1200 AS tripduration, '519' AS start_station_id, '492' AS end_station_id,\n",
    "          'Subscriber' AS usertype, 8 AS hora_inicio, 2 AS dia_semana))\n",
    "\"\"\")"
   ],
   "id": "c12"
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined